# 1.4 分词器类：编码与解码

训练得到的 `vocab` 和 `merges` 需要封装成一个分词器：`encode` 把文本转换为 token ID，`decode` 把 token ID 还原为文本。

In [8]:
import os
import sys

# 直接指定项目根目录，后面的路径一律相对它来写；需替换成自己实际的项目根目录的绝对路径
os.chdir(r"C:\Users\intangible\Desktop\easy-llm")

# 注意切目录并不会让 import 找到 scripts/ 下的模块，还得把它加进模块搜索路径
sys.path.insert(0, os.path.join(os.getcwd(), "scripts"))

print("工作目录:", os.getcwd())
# 工作目录: C:\Users\intangible\Desktop\easy-llm

工作目录: C:\Users\intangible\Desktop\easy-llm


## 核心实现

分词器只负责加载词表、按已有合并规则编码，以及将 token ID 解码为文本。

In [9]:
import regex as re

# GPT-2 预分词正则：把文本切成英文缩写、字母、数字、标点/符号和空白片段。
# BPE 的合并只在单个片段内部发生
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""


In [11]:
# 把 ids 中「所有相邻的 pair」都替换成新 ID idx
def _merge(ids, pair, idx):
    newids = []
    i = 0
    while i < len(ids):
        # 命中 pair 就替换成合并后的 idx，并跳两个位置；否则原样保留、跳一个位置。
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids


In [12]:
class Tokenizer:
    def __init__(self, vocab=None, merges=None, special_tokens=None):
        self.vocab = vocab or {} # token id -> token 字节序列的映射
        self.merges = merges or [] # 合并对 (p0, p1)，p0 和 p1 均为字节序列形式 
        
        # merge_ranks 记录每条 merge 的「创建顺序」（rank）。
        # rank 越靠前/越小 = 训练时越早被合并 = 全局语料里越高频，编码时优先合并它，与训练保持一致
        self.merge_ranks = {pair: rank for rank, pair in enumerate(self.merges)} # 合并对 (p0, p1)，p0 和 p1 均为字节序列形式 

        self.special_tokens = list(special_tokens) if special_tokens else []
        self.vocab_rev = {token: idx for idx, token in self.vocab.items()} # token 字节序列 -> token id 的映射，后续自有妙用

        escaped = [re.escape(token) for token in sorted(self.special_tokens, key=len, reverse=True)] # 对特殊 token 按「长度降序」排序（保证 "<|endoftext|> <|endoftext|>" 先识别）
        self.pattern = "|".join(escaped) if escaped else r"(?!)" # 和训练时构造的特殊 token 正则表达式一致

    def encode(self, text: str) -> list[int]:
        encoded_ids = [] # 最终返回的 id 序列

        # 第一步：按特殊 token 切分。"(" + pattern + ")" 把特殊 token 保留在结果里；训练时不保留，编码时必须保留，要让模型看到特殊 token，学习它们的含义
        for chunk in re.split("(" + self.pattern + ")", text):
            
            if chunk in self.special_tokens: # 当前块本身是特殊 token：先转换成字节序列 bytes，再查反向词表取 ID
                encoded_ids.append(self.vocab_rev[chunk.encode("utf-8")])
                continue
            # 第二步：普通片段再用 GPT-2 正则切成 piece，piece 内部独立做 BPE 合并
            for piece in re.findall(PAT, chunk):
                byte_seq = piece.encode("utf-8")

                # list(byte_seq)
                # 每个字节先映射成基础 token ID；单个字节的 id 一般属于 0~255，但也可能偏移到 offset~offset+255（比如先加 offset 个特殊 token，再加基础字节）
                ids = [self.vocab_rev[byte_seq[i:i + 1]] for i in range(len(byte_seq))] 
                
                while len(ids) > 1: # 反复选「合并优先级最高」的相邻对并合并，直到只剩一个或没有可合并的 pair
                    candidates = set(zip(ids, ids[1:])) # 拿到所有的相邻 pair（无重复）
                    
                    # 用 merge_ranks 查优先级，找到本轮需要合并的那一个 pair；某一 pair 优先级为 inf，说明不在 merges 规则里，即「永远不该被合并」
                    pair = min(
                        candidates,
                        key=lambda p: self.merge_ranks.get(
                            (self.vocab[p[0]], self.vocab[p[1]]), float("inf") # 需要转换成字节序列形式，对齐 merge_ranks 里面的键
                        ),
                    )

                    if pair not in self.merge_ranks:
                        break 
                    
                    # 先将 pair 拼接成字节序列，再转换成 merge_id
                    merged_bytes = self.vocab[pair[0]] + self.vocab[pair[1]]
                    merge_id = self.vocab_rev.get(merged_bytes)
                    
                    ids = _merge(ids, pair, merge_id) # 从左往右将 ids 中的相邻 pair 合并成 merge_id

                encoded_ids.extend(ids) # 将完全合并后的 ids 中的元素全部加入 encoded_ids 末尾：[1, 2] + [3, 4] = [1, 2, 3, 4]
        
        return encoded_ids

    def decode(self, ids: list[int]) -> str:
        # 比 encode 简单多了：将所有 id 转换成字节序列后拼接在一起得到 bytes 对象，再使用自带的 decode() 方法解码回原始字符串
        token_bytes = b"".join(self.vocab[idx] for idx in ids)

        # errors="replace"：遇到不完整的非法字节序列时输出 U+FFFD；方便调试模型偶尔生成的非法 token 组合
        return token_bytes.decode("utf-8", errors="replace")


## 加载真实词表验证

完整实现还包含 `from_files` 和 `encode_iterable`，下面加载 TinyStories tokenizer 做验证。

In [13]:
# 加载完整 Tokenizer，验证真实 TinyStories 词表。
import json
from bpe_tokenizer import Tokenizer as FullTokenizer

vocab_path = "results/tokenizer/TinyStoriesV2-GPT4-train/vocab.json"
with open(vocab_path, encoding="utf-8") as f:
    raw_vocab = json.load(f)
special_token = bytes.fromhex(raw_vocab["256"]).decode("utf-8")

tok = FullTokenizer()
tok.from_files(
    vocab_path,
    "results/tokenizer/TinyStoriesV2-GPT4-train/merge.txt",
    [special_token],
)

s = "Once upon a time, there was a little girl"
ids = tok.encode(s)
print("原文:", repr(s))
print("编码:", ids)
print("解码:", repr(tok.decode(ids)))
print("还原一致:", tok.decode(ids) == s)
# 原文: 'Once upon a time, there was a little girl'
# 编码: [437, 446, 259, 403, 44, 407, 283, 259, 397, 485]
# 解码: 'Once upon a time, there was a little girl'
# 还原一致: True

# 特殊 token：整体占一个 ID，解码时完整还原
s2 = "hello" + special_token + "world"
ids2 = tok.encode(s2)
print("\n含特殊 token 原文:", repr(s2))
print("编码:", ids2)
print("解码:", repr(tok.decode(ids2)))
# 含特殊 token 原文: 'hello' + special_token + 'world'
# 编码: [5659, 256, 119, 304, 342]
# 解码结果与输入一致


原文: 'Once upon a time, there was a little girl'
编码: [437, 446, 259, 403, 44, 407, 283, 259, 397, 485]
解码: 'Once upon a time, there was a little girl'
还原一致: True

含特殊 token 原文: 'hello<|end_of_text|>world'
编码: [5659, 256, 119, 304, 342]
解码: 'hello<|end_of_text|>world'


In [10]:
# 不完整的 UTF-8 字节序列会被 errors="replace" 替换为 U+FFFD 替换字符（�）。
bad_bytes = b"\xe6\x88\xe6\x88\xe6\x88"           # '你' 的前两个字节重复三遍，单独解码非法
decoded = bad_bytes.decode("utf-8", errors="replace")
print(decoded)                      # 直接打印，替换字符 � 原样显示
# -> �

���
